# ceol-gpt — Colab Training Notebook

**Before running:** In the Colab menu go to `Runtime → Change runtime type` and select **A100 GPU** or **H100 GPU**.

This notebook:
1. Mounts Drive and clones / pulls the repo
2. Installs dependencies
3. Verifies the GPU and tokenizer
4. Runs training (with automatic resume if the session drops)
5. Plots training curves from the saved log
6. Runs a quick generation sanity-check from the best checkpoint

## 0 — Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Edit this path to wherever you keep the repo in your Drive ───────────────
REPO_PATH = '/content/drive/MyDrive/Duke/cs372/Project/ceol-gpt'
GITHUB_URL = 'https://github.com/andrewmcknight/ceol-gpt.git'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(REPO_PATH):
    print('Repo already exists — pulling latest changes')
    !git -C {REPO_PATH} pull
else:
    print('Cloning repo into Drive')
    !git clone {GITHUB_URL} {REPO_PATH}

%cd {REPO_PATH}
import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

!ls

## 1 — Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU found! Go to Runtime → Change runtime type.'

device_name = torch.cuda.get_device_name(0)
vram_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
cap         = torch.cuda.get_device_capability()
amp_dtype   = 'bfloat16' if cap[0] >= 8 else 'float16'

print(f'GPU:      {device_name}')
print(f'VRAM:     {vram_gb:.1f} GB')
print(f'Compute:  sm_{cap[0]}{cap[1]}')
print(f'AMP:      {amp_dtype}')

# Recommend config based on GPU
if 'H100' in device_name or 'A100' in device_name:
    print('\n→ Use configs/large.yaml  (47M params, recommended)')
elif 'L4' in device_name:
    print('\n→ Use configs/medium.yaml  (12M params)')
else:
    print('\n→ Use configs/small.yaml  (2M params, T4/other)')

## 3 — Build / verify tokenizer

In [ ]:
import json
from pathlib import Path
from src.tokenizer import ABCTokenizer

TOK_PATH = 'models/tokenizer.pkl'

if Path(TOK_PATH).exists():
    tokenizer = ABCTokenizer.load(TOK_PATH)
    print(f'Loaded tokenizer from {TOK_PATH}')
else:
    print('Building tokenizer from scratch (takes ~30s)...')
    with open('data/tunes.json') as f:
        tunes = json.load(f)
    tokenizer = ABCTokenizer.from_tunes(tunes, min_freq=2)
    Path('models').mkdir(exist_ok=True)
    tokenizer.save(TOK_PATH)
    print(f'Saved to {TOK_PATH}')

print(f'Vocab size: {len(tokenizer):,}')

## 4 — Choose config & run training

Set `CONFIG` to whichever config matches your GPU (see step 2).

If your session disconnects and restarts, **just re-run all cells** — training resumes automatically from the latest checkpoint thanks to `--resume`.

In [ ]:
CONFIG   = 'configs/large.yaml'   # large / medium / small
RUN_NAME = 'large'                # checkpoints saved to models/{RUN_NAME}/

# NUM_WORKERS=2 is safe on Colab; increase to 4 on high-RAM runtimes
import os
os.environ['NUM_WORKERS'] = '2'

print(f'Config:   {CONFIG}')
print(f'Run name: {RUN_NAME}')
print(f'Checkpoints will be saved to: models/{RUN_NAME}/')

In [ ]:
# This cell can be re-run after a disconnect — training resumes from latest.pt
!python -m src.train --config {CONFIG} --run-name {RUN_NAME} --resume

## 5 — Plot training curves

Can be run at any time during or after training — just re-run the cell.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(f'models/{RUN_NAME}/train_log.jsonl')
assert log_path.exists(), f'Log not found at {log_path} — has training started?'

entries = [json.loads(line) for line in log_path.read_text().strip().splitlines()]
epochs      = [e['epoch']      for e in entries]
train_loss  = [e['train_loss'] for e in entries]
val_loss    = [e['val_loss']   for e in entries]
lrs         = [e['lr']         for e in entries]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, train_loss, label='Train loss')
ax1.plot(epochs, val_loss,   label='Val loss')
best_epoch = epochs[val_loss.index(min(val_loss))]
ax1.axvline(best_epoch, color='green', linestyle='--', alpha=0.6, label=f'Best (ep {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Training curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, lrs)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning rate')
ax2.set_title('LR schedule (warmup + cosine decay)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'models/{RUN_NAME}/training_curves.png', dpi=150)
plt.show()

print(f'Epochs completed: {len(epochs)}')
print(f'Best val loss:    {min(val_loss):.4f}  (epoch {best_epoch})')
print(f'Final train loss: {train_loss[-1]:.4f}')

## 6 — Quick generation sanity-check

Loads the best checkpoint and generates one tune. This is just a smoke-test to confirm the checkpoint is valid — a proper generation script (`src/generate.py`) comes later.

In [ ]:
import torch
import yaml
from src.model import build_model
from src.tokenizer import ABCTokenizer

# ── Generation parameters ────────────────────────────────────────────────────
GEN_TYPE        = 'reel'
GEN_KEY         = 'Gmajor'
GEN_METER       = '4/4'
TEMPERATURE     = 0.8
TOP_K           = 50
MAX_NEW_TOKENS  = 256
# ─────────────────────────────────────────────────────────────────────────────

device    = torch.device('cuda')
ckpt_path = f'models/{RUN_NAME}/best.pt'

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

tokenizer = ABCTokenizer.load(f'models/{RUN_NAME}/tokenizer.pkl')
model     = build_model(cfg, vocab_size=len(tokenizer)).to(device)
ckpt      = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"] + 1}, val loss {ckpt["val_loss"]:.4f}')

In [ ]:
import torch.nn.functional as F

def generate(
    model, tokenizer, tune_type, key, meter,
    temperature=0.8, top_k=50, max_new_tokens=256,
):
    """Simple autoregressive sampling (top-k). Full generation in src/generate.py."""
    # Seed with conditioning prefix + BOS
    prompt_ids = tokenizer.encode('', tune_type, key, meter, add_eos=False)
    ids = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(ids)               # (1, T, V)
            logits = logits[:, -1, :]         # last position: (1, V)

            # Top-k filtering
            if top_k > 0:
                vals, _ = torch.topk(logits, top_k)
                logits[logits < vals[:, -1:]] = float('-inf')

            probs  = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

            if next_id.item() == tokenizer.eos_id:
                break
            ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode_to_abc(ids[0].tolist())


abc = generate(
    model, tokenizer,
    tune_type=GEN_TYPE, key=GEN_KEY, meter=GEN_METER,
    temperature=TEMPERATURE, top_k=TOP_K, max_new_tokens=MAX_NEW_TOKENS,
)

print(f'X:1')
print(f'T:Generated {GEN_TYPE}')
print(f'M:{GEN_METER}')
print(f'L:1/8')
print(f'K:{GEN_KEY}')
print(abc)